# 🏰 Virtual Environments — Your Own Kitchen for Every Project

Up to this point, everything you installed went into one global, shared space. For a few small experiments that works fine — but the moment you hold two real projects, the global space turns into a war zone. Project A needs `cheese_lib` v1, Project B needs the same-named `cheese_lib` v2, and a single shared folder can only hold one of them. Whichever installs last wins; the other project silently breaks. This is the classic dependency conflict, and the professional solution is the **virtual environment**: a private, isolated folder holding its own Python, its own `pip`, and its own libraries, completely separate from the system's global installation.

## 1. What you will learn here

1. **Why environments matter** — the dependency-isolation story, told through the kitchen war of two projects.
2. **The 3-step terminal mantra** — create → activate → work inside; leave with `deactivate`.
3. **`requirements.txt` — the project passport** — export installed packages with `pip freeze`, revive them anywhere with `pip install -r requirements.txt`.
4. **The venv anatomy** — a live `python -m venv` demo that shows you `bin/`, `lib/`, `pyvenv.cfg`, and the activation scripts.
5. **The isolation proof** — `sys.prefix != sys.base_prefix` printed as `True`, the mechanical formula that guarantees your env is genuinely separate.

## 2. How the demos work

The notebook runs real `venv` commands from inside Python using `subprocess`, building a throwaway environment in a temp folder. Because interacting with a shell `activate` script mid-notebook is awkward, the demos call the environment's own `python` binary directly — which is exactly what activation does behind the scenes. You will also get copy-paste cheat-sheets for both Linux/macOS and Windows terminals, because in everyday life you will `venv` from a shell, not a notebook.

## 3. The big idea

Every project gets its own kitchen: separate counters, separate libraries, separate pip life. Two projects needing conflicting versions coexist peacefully because each sees only its own kitchen. This one idea removes an entire class of “works on my machine” disasters and is standard practice in every Python workplace.

## 🍳 Real-World Hook: The Kitchen-War of 2 Projects

Project-A (a pizza app 🍕) needs `cheese_lib` v1; Project-B (a salad dash 🥗) needs `cheese_lib` v2 with the same name. In ONE global microwave (global pip!) only one can be installed — a nasty tug-of-war: whoever installs last is the winner, and the other project breaks.

**The root solution:** a **separate kitchen** for every project. Separate counters, separate lib-versions, separate pip-life — no war. A virtual environment is exactly that kitchen: an isolated folder holding its own `python`, its own `pip`, and its own libraries, completely locked away from the base system.

### 📖 Definition (interview-grade)

> **Virtual Environment** = an isolated folder that owns its own `python`, its own `pip`, and its own installed libraries, fully separated from the system-level installation. `activate` = the entry pass 🎫 that makes your shell use that folder's tools; `deactivate` = leaving the kitchen.

The live demo next will build one of these kitchens and inspect its rooms.

# 1. Creating a Kitchen — `venv` Anatomy
The venv concept is simple; the folder it creates is a small, organised world. The next cell runs `python -m venv <folder>` and then inspects what was built.

## 1.1 What `python -m venv` creates

`python -m venv myenv` builds a new self-contained Python world inside the folder `myenv`. Inside it you will typically find:

| Entry | Purpose |
|---|---|
| `bin/` (Linux/macOS) or `Scripts/` (Windows) | launcher scripts: your own `python`, `pip`, plus the activation scripts (`activate`, `Activate.ps1`, `activate.csh`) |
| `lib/` (or `Lib/`) | the private site-packages where your env's libraries get installed — isolated from the global site-packages |
| `include/` | C-extension headers used when building packages with native code |
| `pyvenv.cfg` | the config card pointing back to the base (system) Python that spawned this env |
| `.gitignore` | a polite helper so you do not accidentally commit the heavy env folder (newer versions add it automatically) |

The demos use a temp directory for the environment, but in a real project you would create `myenv` right inside the project folder. The activation scripts in `bin/` are the magic behind the `(myenv)` prompt you will see in terminals — that prompt is the signal that your shell commands now run inside this kitchen.

The expected output in the cell shows exactly this layout (`bin`, `include`, `lib`, `pyvenv.cfg` — plus `.gitignore` on this Python version), and a peek into `bin/` reveals the launcher and activation files. That is the whole anatomy: interpreters, launchers, config, and an isolated library shelf.

In [1]:
# 🐣 STEP-1: venv ka anatomy (folder structure-preview!) 🗂️
import sys, os, subprocess, tempfile

BASE = tempfile.mkdtemp()
ENV = os.path.join(BASE, "demov")
print("creating venv at:", ENV, "(takes ~10-20s...)")
subprocess.run([sys.executable, "-m", "venv", ENV], check=True)
print("\n📂 venv-internal folders:", sorted(os.listdir(ENV)))
print("🔧 venv/bin sneak-peek   :", sorted(os.listdir(os.path.join(ENV, "bin")))[:5])
# Expected OUTPUT:
# creating venv at: /tmp/... (takes ~10-20s...)
#
# 📂 venv-internal folders: ['bin', 'include', 'lib', 'lib64', 'pyvenv.cfg']
# 🔧 venv/bin sneak-peek   : ['Activate.ps1', 'activate', 'activate.csh', 'pip', 'pip3']

creating venv at: /tmp/tmpj9lg8hux/demov (takes ~10-20s...)

📂 venv-internal folders: ['.gitignore', 'bin', 'include', 'lib', 'lib64', 'pyvenv.cfg']
🔧 venv/bin sneak-peek   : ['Activate.ps1', 'activate', 'activate.csh', 'activate.fish', 'pip']


## 1.2 Proof of isolation — `sys.prefix` vs `sys.base_prefix`

How do we *prove* an environment is actually isolated rather than just a folder with a nice name? We ask the environment's own Python two questions and compare the answers.

- `sys.base_prefix` — the location of the base (system) Python that created the environment.
- `sys.prefix` — the location of the Python that is currently running.

Inside a virtual environment these two differ: `sys.prefix` points into your env folder while `sys.base_prefix` still points at the system Python. The check `sys.prefix != sys.base_prefix` therefore prints **`True`** — the mechanical proof that `demov/bin/python` is running inside its own isolated world. Outside an environment, in the base interpreter, both values are equal and the check prints `False`.

The cell also runs `pip list` from the env's own pip: a fresh environment contains almost nothing (just pip and setuptools on classic builds) — the clean slate that guarantees no global packages leak into your kitchen. Two numbers (`prefix` mismatch) and a short package list tell you everything: interpreters differ, libraries are empty — you are truly isolated.

The anatomy note after the demo reminds you how Windows mirrors everything under `Scripts/\activate` instead of `bin/activate`, and why keeping envs out of git is the industry rule.

In [2]:
# 🔬 STEP-2: ISOLATION-PROOF — venv-python hi apna-world! 🎫
pybin = os.path.join(ENV, "bin", "python")

ver = subprocess.run([pybin, "-V"], capture_output=True, text=True).stdout.strip()
print("venv-python :", ver)

proof = subprocess.run([pybin, "-c",
    "import sys; print(sys.prefix != sys.base_prefix)"],
    capture_output=True, text=True).stdout.strip()
print("isolated?🎫:", proof, "(venv-prefix ≠ base-prefix!)")

# venv-pip list — tiny-fresh-start!
pl = subprocess.run([os.path.join(ENV, "bin", "pip"), "list", "--format=freeze"],
                    capture_output=True, text=True).stdout.strip()
print("venv-pkgs   :", len(pl.splitlines()) if pl else 0, "packages (sirf pip/setuptools-fresh-start! 🌱)")
# Expected OUTPUT:
# venv-python : Python 3.13.14 (aapke-wala!)
# isolated?🎫: True (venv-prefix ≠ base-prefix!)
# venv-pkgs   : 2 packages (sirf pip/setuptools-fresh-start! 🌱)

venv-python : Python 3.13.14
isolated?🎫: True (venv-prefix ≠ base-prefix!)
venv-pkgs   : 1 packages (sirf pip/setuptools-fresh-start! 🌱)


### 💡 Anatomy-amusements:
- `bin/` → activate-scripts + your own `python`+`pip` (Linux/Mac) 🐧
  (on Windows the folder = `Scripts/` + `activate.bat`! 🪟)
- `lib/` → all the libraries of this kitchen (isolated-racks!)
- `pyvenv.cfg` → the config-card (the base-python linker!)
> Notice Step-2's `True` — **venv-prefix ≠ base-prefix** = the isolation
> formula! This is the THE-END pass for project-wars! ✌️


## 🧘 Terminal-CASTLE — The 3-Step Command Mantra!

### 🐧 Linux/Mac:
```bash
python3 -m venv myenv              # 1) kitchen-build (naye-folder!)
source myenv/bin/activate          # 2) ENTRY-pass 🎫 (prompt-start: (myenv)
python -V; which pip               # 3) INSIDE-proofs — venv-python vad-hi!
pip install numpy pandas           #    libs isi-kitchen me!
pip freeze > requirements.txt      #    passport-export 📋
deactivate                         # 4) EXIT! (prompt normal!)
```

### 🪟 Windows (cmd / PowerShell):
```bat
python -m venv myenv
myenv\Scripts\activate          :: cmd-style
myenv\Scripts\Activate.ps1      :: PowerShell-style (run-policy care!)
python -V
pip install jupyter
deactivate
```

### 📋 Revive a project on a new machine (the Requirements passport!):
```bash
python3 -m venv myenv && source myenv/bin/activate
pip install -r requirements.txt    ← purane-recipe saari-libs back! 🍳
```


## ⚖️ Venv-usage rules — Golden-Rules!

| Rule | Why |
|---|---|
| 1. venv-folder NOT in GIT (.gitignore!) | heavy folder + machine-specific! |
| 2. keep only `requirements.txt` in the repository! | recipe-passport (small + portable!) 📋 |
| 3. your own env for every project | dependency-war THE-END ✌️ |
| 4. pip-install only after activate | otherwise global-pollution! |
| 5. `deactivate` when you exit | prompt safe, world safe! 🎫 |

> 🪤 **Common-mistake:** tell your editor/IDE about the venv too (VSCode: Ctrl+Shift+P →
> "Python: Select Interpreter" → ./myenv/bin/python!) — otherwise the editor complains
> "module not found!" even though it was pip-installed! 🎭


## Summary

Virtual environments isolate every project's dependencies so conflicting library versions never fight. A venv is a self-contained folder with its own `python`, `pip`, and libraries in `bin/` (or `Scripts/` on Windows), `lib/`, and `pyvenv.cfg`. The 3-step mantra: create (`python -m venv myenv`), activate (`source myenv/bin/activate`; on Windows `myenv\Scripts\activate`), and leave (`deactivate`). Isolation is provable — inside the env, `sys.prefix != sys.base_prefix` prints `True`, and a fresh `pip list` shows an almost-empty shelf. The project passport is `requirements.txt`: export with `pip freeze > requirements.txt`, revive on another machine with `pip install -r requirements.txt`. Golden rules: keep env folders out of git (the recipe `requirements.txt` belongs in the repo), pip-install only after activating, and point your editor's interpreter at the env. Next stop: conda and uv — two more environment power tools.